In [ ]:
import igraph as ig
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import networkx as nx
import multiprocessing

rng = np.random.default_rng()

In [ ]:
def simulate(graph, window=0, n_probs=100, copies=1000, low_p=0.4, high_p=0.6, no_bar=False):
    n_edges = graph.ecount()
    lcc = np.zeros(n_probs, dtype="float32")
    all_prob = np.linspace(low_p, high_p, n_probs)
    for idx, p in tqdm(enumerate(all_prob), total=n_probs, disable=no_bar):
        is_removed = rng.random((copies, n_edges))
        link_prob = rng.uniform(
            max(p - window, 0), min(p + window, 1), (copies, n_edges)
        )
        # link_prob = np.repeat(link_prob, copies, axis=0)
        links_removed = is_removed >= link_prob
        for rep in range(copies):
            links_to_remove = np.flatnonzero(links_removed[rep])
            work = graph.copy()
            work.delete_edges(links_to_remove)
            lcc[idx] += max(work.components().sizes())
        lcc[idx] /= copies
    return all_prob, lcc / graph.vcount()

In [192]:
g = ig.Graph.Lattice([20, 20], circular=False)
lcc = []
windows = np.linspace(0, 0.5, 11)
for win in tqdm(windows):
    prob, temp = simulate(g, window=win, no_bar=True, n_probs=50, copies=100)
    lcc.append(temp)

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
g = nx.grid_2d_graph(3, 3)
plt.figure(figsize=(8, 2))
pos = {(x, y): (y, -x) for x, y in g.nodes()}
nx.draw(g, pos=pos, node_size=100)

In [ ]:
n_edges = g.number_of_edges()
arr = rng.integers(2, size=n_edges, dtype="bool")
arr

In [ ]:
g.remove_edges_from([e for e, truth in zip(g.edges, arr) if truth])

In [ ]:
pos = {(x, y): (y, -x) for x, y in g.nodes()}
nx.draw(g, pos=pos, node_size=100)

In [ ]:
len(max(nx.connected_components(g), key=len))

In [ ]:
def simulate_nx(graph: nx.Graph, window=0, n_probs=100, copies=1000, low_p=0.4, high_p=0.6, no_bar=False):
    n_edges = graph.number_of_edges()
    lcc = np.zeros(n_probs, dtype="float32")
    all_prob = np.linspace(low_p, high_p, n_probs)
    for idx, p in tqdm(enumerate(all_prob), total=n_probs, disable=no_bar):
        is_removed = rng.random((copies, n_edges))
        link_prob = rng.uniform(
            max(p - window, 0), min(p + window, 1), (copies, n_edges)
        )
        # link_prob = np.repeat(link_prob, copies, axis=0)
        links_removed = is_removed >= link_prob
        for rep in range(copies):
            work = graph.copy()
            work.remove_edges_from([e for e, truth in zip(g.edges, links_removed[rep]) if truth])
            lcc[idx] += len(max(nx.connected_components(work), key=len))
        lcc[idx] /= copies
    return all_prob, lcc / graph.number_of_nodes()

In [ ]:
g = nx.grid_2d_graph(20, 20)
lcc = []
windows = np.linspace(0, 0.5, 11)
for win in tqdm(windows):
    prob, temp = simulate_nx(g, window=win, no_bar=True, n_probs=50, copies=100)
    lcc.append(temp)

In [193]:
def multi_simulation(win):
    g = nx.grid_2d_graph(20, 20)
    return simulate_nx(g, win, no_bar=False, n_probs=50, copies=100)

In [194]:
points = 11
with multiprocessing.Pool() as pool:
    windows = np.linspace(0, 0.5, points)
    results = pool.imap(multi_simulation, windows)

    lcc = []
    for prob, temp in results:
        lcc.append(temp)
# np.savetxt("results.txt")